# 🧠 Etapa 1: Entrenamiento Distribuido y Fine-Tuning de BETO

Este notebook constituye el núcleo del pipeline de **MLOps**. Su función es tomar los datos crudos del almacenamiento de objetos, realizar el entrenamiento del modelo de lenguaje y exportar los pesos resultantes de forma que puedan ser consumidos por los nodos posteriores.

---

### 🛠️ 1. Preparación y Resiliencia del Entorno
Dado que el pipeline corre en nodos efímeros de **OpenShift AI**, implementamos una lógica de preparación robusta:
* **Bootstrap de Librerías:** Instalamos versiones específicas de `transformers`, `accelerate` y `optimum` para garantizar compatibilidad.
* **Recarga de Site-Packages:** Forzamos a Python a reconocer las instalaciones nuevas sin reiniciar el kernel.
* **Gestión de Permisos (utime):** Establecemos el directorio de trabajo en `/opt/app-root/src` para evitar errores de escritura comunes en entornos restringidos de Kubernetes.

### 📥 2. Conectividad y Extracción de Datos
Establecemos un puente con **MinIO** para descargar el dataset generado en la etapa anterior. 
* Se utiliza el protocolo S3 para garantizar que el entrenamiento sea agnóstico a la ubicación del dato.
* Se implementa un **Split automático** para separar los datos en conjuntos de Entrenamiento (80%) y Validación (20%).

### 🚀 3. Fine-Tuning del Modelo BETO
Entrenamos el modelo **BERT-base-spanish** (BETO) ajustándolo para las 4 intenciones de negocio de **Movistar**:
* **Tokenización Dinámica:** Conversión de texto a vectores con truncamiento a 128 tokens para optimizar la memoria.
* **Precisión Mixta (FP16):** Activación automática de entrenamiento acelerado si se detecta una GPU disponible en el clúster.
* **Configuración de Trainer:** Optimización de hiperparámetros para un entrenamiento rápido y eficiente en contenedores.

### 📦 4. Empaquetado de Artefactos (Pipeline Readiness)
Para que el modelo sobreviva a la transición entre pods de **Elyra**, realizamos dos acciones finales:
1. **Serialización:** Guardamos los pesos y la configuración del tokenizador.
2. **Compresión (`.tar.gz`):** Creamos un paquete comprimido que contiene todo el "cerebro" de la IA, facilitando su transporte hacia el nodo de optimización (ONNX).

---
> **Estado:** Al finalizar, se genera el archivo `modelo_pytorch.tar.gz`, el cual es el insumo principal para la etapa de conversión y despliegue.

In [1]:
import sys
import subprocess
import os

def install_packages():
    packages = [
        "transformers[torch]>=4.40.0", 
        "accelerate>=0.26.0", 
        "datasets", 
        "boto3", 
        "optimum[onnxruntime]",
        "pyarrow"
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U"] + packages)

install_packages()

import site
from importlib import reload
reload(site)

import torch
import transformers
import accelerate
print(f"Transformers version: {transformers.__version__}")
print(f"Accelerate version: {accelerate.__version__}")

# home del usuario
working_dir = "/opt/app-root/src"
os.chdir(working_dir)

import pandas as pd
import numpy as np
import boto3
from botocore.client import Config
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer


s3_endpoint = os.getenv("S3_ENDPOINT", "http://minio-service.t-moviles-ai.svc.cluster.local:9000")
s3_access_key = os.getenv("S3_ACCESS_KEY", "minio")
s3_secret_key = os.getenv("S3_SECRET_KEY", "minio123")
bucket_name = "training-data"
file_name = "t-moviles_dataset.csv"

s3 = boto3.client(
    's3',
    endpoint_url=s3_endpoint,
    aws_access_key_id=s3_access_key,
    aws_secret_access_key=s3_secret_key,
    config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
    verify=False,
    region_name='us-east-1'
)

print(f"Descargando {file_name}...")
s3.download_file(bucket_name, file_name, file_name)

dataset = load_dataset('csv', data_files=file_name)
dataset = dataset['train'].train_test_split(test_size=0.2)

model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_func, batched=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

training_args = TrainingArguments(
    output_dir="./results_t-moviles",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4, 
    num_train_epochs=3,
    weight_decay=0.01,
    # No guardamos checkpoints
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"]
)

print("🚀 Iniciando entrenamiento en OpenShift AI...")
trainer.train()


import tarfile
import os

model_path = "modelo_t-moviles_pytorch"
output_tar = "modelo_pytorch.tar.gz"

# Guardar el modelo
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

# Archivo .tar.gz
print(f"📦 Empaquetando modelo en {output_tar}...")
with tarfile.open(output_tar, "w:gz") as tar:
    tar.add(model_path, arcname=os.path.basename(model_path))

print("✅ Archivo listo para el siguiente step.")

INFO: pip is looking at multiple versions of optimum-onnx[onnxruntime] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of optimum-onnx[onnxruntime] to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of transformers[torch] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers[torch] to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constr

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/app-root/lib64/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


🚀 Iniciando entrenamiento en OpenShift AI...


Epoch,Training Loss,Validation Loss
1,No log,0.009986


✅ Proceso completado. Modelo en: modelo_t-moviles_pytorch
